# 🗓️ 25일차 스터디 노트북 — 도수 정렬 (비교하지 않는 정렬)

**오늘 범위**: 06-9 도수 정렬(counting sort) — 1단계 도수 분포표 · 2단계 누적 도수 분포표 · 3단계 작업용 배열 · 4단계 복사 · 실습 6-17 · 안정성과 제약

## 난이도 태그
🟢 **기본** (전원 필수) / 🟡 **표준** (팀 목표선) / 🔴 **심화** (도전)

## 유형 태그
**[손]** · **[빈칸]** · **[예측]** · **[구현]** · **[디버깅]** · **[설명]** · **[실험]** · **[추적]**

---

## 오늘의 한 문장

> **"지금까지 배운 정렬은 전부 `if a[i] > a[j]` 를 갖고 있었다. 그걸 지우면 어떻게 될까?"**

교재 297p 첫 줄: **"도수 정렬은 원소의 대소 관계를 판단하지 않고 빠르게 정렬하는 알고리즘"**

| | 원소 비교 | 시간 복잡도 |
|---|---|---|
| 버블·선택·삽입 (18~19일) | ✅ | O(n²) |
| 셸·퀵·병합·힙 (20~24일) | ✅ | O(n log n) |
| **도수 (오늘)** | ❌ **안 함** | **O(n + max)** |

비교 기반 정렬의 이론적 하한이 O(n log n)이야. 도수 정렬은 **비교를 안 하기 때문에** 그 벽을 뚫어. 대신 **조건**이 붙지 — 그게 뭔지가 오늘의 후반부야.

## 오늘의 진행 순서

**개념(1~2) → 1단계 도수표(3~4) → 2단계 누적(5~7) → 3단계 작업배열(8~11) → 4단계(12) → 코드(13~17) → 성질과 제약(18~20)**

표를 **직접 채우는** 문제가 많아. 종이랑 펜 준비.

> 📁 아래 **"부록"** 셀을 **가장 먼저 실행**해.

---

## 📁 부록 — 오늘의 도구 (제일 먼저 실행!)

In [ ]:
from typing import MutableSequence
import random, time, tracemalloc

def show_arr(arr, name="", mark=None, width=4):
    """배열을 인덱스와 함께 표 형태로 출력. mark=강조할 인덱스 집합"""
    idx = "".join(f"{i:^{width}}" for i in range(len(arr)))
    val = "".join((f"[{v}]".center(width) if mark and i in mark else f"{'' if v is None else v}".center(width))
                  for i, v in enumerate(arr))
    print(f"  {name:>4s} idx {idx}")
    print(f"  {'':>4s} val {val}")

# 교재 실습 6-17
def fsort(a: MutableSequence, max: int) -> None:
    """도수 정렬(배열 원솟값은 0이상 max 이하)"""
    n = len(a)
    f = [0] * (max + 1)          # 누적 도수 분포표 배열 f
    b = [0] * n                  # 작업용 배열 b

    for i in range(n):           # [1단계] 도수 분포표
        f[a[i]] += 1
    for i in range(1, max + 1):  # [2단계] 누적 도수 분포표
        f[i] += f[i - 1]
    for i in range(n - 1, -1, -1):   # [3단계] 작업용 배열 만들기
        f[a[i]] -= 1; b[f[a[i]]] = a[i]
    for i in range(n):           # [4단계] 배열 복사
        a[i] = b[i]

def counting_sort(a: MutableSequence) -> None:
    """도수 정렬"""
    if len(a) > 0:
        fsort(a, max(a))

# 오늘 계속 쓸 교재 예제 (그림 6-38)
SAMPLE = [5, 7, 0, 2, 4, 10, 3, 1, 3]

print("준비 완료 ✅")
show_arr(SAMPLE, "a")
print(f"\n  n = {len(SAMPLE)}, max = {max(SAMPLE)}")

---
# 🔁 [Remind] 워밍업 — 되감기

오늘은 **비교 정렬 전체**와 **안정성**을 소환해.

### R-1. 🟢 [설명] 지금까지의 정렬엔 공통점이 있었다

이번 주에 배운 것들의 핵심 줄을 모아봤어.

```python
if a[j] < a[m]: m = j                       # 선택 (19일)
while j > 0 and a[j-1] > tmp:               # 삽입 (19일)
while a[pl] < x: pl += 1                    # 퀵 (21일)
if buff[j] <= a[i]:                         # 병합 (23일)
if temp >= a[child]: break                  # 힙 (24일)
```

- 다섯 줄의 **공통점**이 뭐야?
- 이런 정렬을 **비교 정렬(comparison sort)** 이라고 해. 비교 정렬의 시간 복잡도는 아무리 잘 만들어도 **O(n log n)보다 빠를 수 없다**는 게 증명되어 있어.
- 💡 그럼 오늘의 질문: **비교를 아예 안 하면?** 그게 가능하려면 원소에 대해 **뭘 미리 알고 있어야** 할까?

### R-2. 🟡 [설명] 안정성 총정리

| 정렬 | 안정? | 왜 |
|---|---|---|
| 삽입 (19일) | ✅ | 같으면 멈춰서 뒤에 삽입 |
| 셸 (20일) | ❌ | h칸씩 건너뛰며 멀리 교환 |
| 퀵 (21일) | ❌ | pl/pr이 먼 원소를 교환 |
| 병합 (23일) | ✅ | `<=` 로 앞쪽 배열 우선 |
| 힙 (24일) | ❌ | 루트와 맨 끝을 교환 |
| **도수 (오늘)** | ①____ | ②____ |

- ①, ②를 **예측만** 해두고 19번에서 확인해.
- 힌트: 도수 정렬은 원소를 **교환하지 않아.** 그럼 안정적일 것 같지?
  ⚠️ 그런데 교재 303p에 **함정**이 하나 적혀 있어. 나중에 만날 거야.

*(여기에 답 작성)*

---
# 🎯 PART 1 — 비교하지 않는다는 것 (1~2번)

### 1. 🟢 [설명] 비교 없이 어떻게 정렬하지?

10점 만점 시험을 본 학생 9명의 점수야.
```
a = [5, 7, 0, 2, 4, 10, 3, 1, 3]
```

**사고 실험**: 만약 선생님이 이 점수들을 **손으로** 정렬한다면, 두 가지 방법이 있어.

**방법 A (비교)**: 답안지 두 장을 집어들고 점수를 비교해서 순서를 바꾼다. → 지금까지 배운 것

**방법 B (칸 나누기)**: 책상에 **0점~10점 칸을 11개** 만들고, 답안지를 하나씩 **해당 칸에 던져 넣는다.** 다 넣은 뒤 0번 칸부터 순서대로 꺼낸다.

- 방법 B에서 답안지끼리 **비교한 적이 있어?**
- 방법 B가 성립하려면 **미리 알아야 하는 것**이 뭐지? (칸을 몇 개 만들어야 하는지 알아야 하잖아)
- 방법 B로 **키(cm)** 를 정렬한다면 칸이 몇 개 필요할까? **연봉**이라면? **실수(3.14 같은)** 라면?
- 💡 교재 297p 각주: **"정렬할 배열은 a, 원소 수는 n, 점수의 최댓값은 max입니다."** → `max`를 알아야 한다는 게 도수 정렬의 **전제 조건**이야.

*(여기에 답 작성)*

### 2. 🟢 [설명] 도수 분포표란

교재 298p "조금만 더": **"도수 분포표는 자료를 몇 개의 등급으로 나누고 각 등급에 속하는 도수를 조사하여 나타낸 표를 의미합니다. 여기에서 도수는 각 등급에 속하는 자료의 개수입니다."**

- "등급"은 우리 문제에서 뭐에 해당해?
- "도수"는?
- 교재 예: **"완성된 도수 분포표에서 f[3]의 값인 2는 3점인 학생이 모두 2명이라는 의미입니다."**
  → `f[7] = 1` 은 무슨 뜻이야? `f[6] = 0` 은?
- 도수 분포표 배열 `f`의 **크기**는 몇이어야 해? 왜 `max`가 아니라 `max + 1`이지?

*(답을 적은 뒤 실행)*

In [ ]:
a = SAMPLE
mx = max(a)
print(f"a = {a}")
print(f"max = {mx} → f의 크기는 max + 1 = {mx + 1}")
print(f"  f의 인덱스는 0 ~ {mx} 까지 총 {mx + 1}개")
print(f"\n만약 크기를 max({mx})로 만들면?")
f_small = [0] * mx
try:
    f_small[mx] += 1
except IndexError as e:
    print(f"  IndexError: {e}  ← 10점 학생을 셀 칸이 없다!")

---
# 1️⃣ PART 2 — 1단계: 도수 분포표 만들기 (3~4번)

> 교재 297p [그림 6-38].

### 3. 🟢 [손] 도수 분포표를 직접 채워보자

```python
for i in range(n):
    f[a[i]] += 1
```

`a = [5, 7, 0, 2, 4, 10, 3, 1, 3]`

**한 칸씩 직접 채워봐.** (교재 그림 6-38의 진행)

| i | a[i] | 하는 일 | f (0~10) |
|---|---|---|---|
| 시작 | | 전부 0 | `[0,0,0,0,0,0,0,0,0,0,0]` |
| 0 | 5 | `f[5] += 1` | `[0,0,0,0,0,1,0,0,0,0,0]` |
| 1 | 7 | `f[7] += 1` | `[_,_,_,_,_,_,_,_,_,_,_]` |
| 2 | 0 | | |
| 3 | 2 | | |
| 4 | 4 | | |
| 5 | 10 | | |
| 6 | 3 | | |
| 7 | 1 | | |
| 8 | 3 | | |

**완성된 도수 분포표 f = `[_, _, _, _, _, _, _, _, _, _, _]`**

- `f[3]`이 **2**가 되는 이유는? (a에서 3이 몇 번 나왔지?)
- `f[6]`, `f[8]`, `f[9]`가 0인 이유는?
- `f`의 **모든 원소를 더하면** 뭐가 나올까? 왜 그럴까?

*(답을 적은 뒤 실행)*

In [ ]:
a = SAMPLE
n = len(a); mx = max(a)
f = [0] * (mx + 1)
print(f"a = {a}\n")
show_arr(f, "f")
for i in range(n):
    f[a[i]] += 1
    print(f"\ni={i}, a[{i}]={a[i]} → f[{a[i]}] += 1")
    show_arr(f, "f", mark={a[i]})

print(f"\n완성된 도수 분포표 f = {f}")
print(f"f의 합 = {sum(f)} = n = {n}  ← 모든 학생이 정확히 한 칸에만 세어지므로")

### 4. 🟡 [설명] `f[a[i]] += 1` — 값이 인덱스가 된다

이 한 줄이 도수 정렬 전체의 핵심 아이디어야.

```python
f[a[i]] += 1
```

- 대괄호가 **두 겹**이지. 안쪽 `a[i]`부터 평가돼. `i=0`일 때 `a[0]`은 5니까 이 줄은 `f[5] += 1`이 돼.
- 지금까지 배운 정렬에서 배열의 **값**은 그냥 데이터였어. 그런데 여기선 값이 **인덱스로 변신**해. 이걸 뭐라고 부를 수 있을까?
- **왜 비교가 필요 없어졌을까?** 값 5는 "다른 원소보다 큰지" 물어볼 필요 없이, 자기가 갈 칸(`f[5]`)을 **스스로 알고 있어**.
- ⚠️ 이 트릭이 성립하려면 값이 어떤 조건을 만족해야 해?
  - 조건 1: ①________ (인덱스로 쓸 수 있어야 하니까)
  - 조건 2: ②________ (배열 범위를 벗어나면 안 되니까)
- `a = [5, -3, 2]` 처럼 **음수**가 있으면 어떻게 될까? 실행해서 확인해봐. **에러가 날까, 조용히 틀릴까?** 🔥

*(예측을 적은 뒤 실행)*

In [ ]:
print("[음수가 있으면?]")
a_neg = [5, -3, 2]
f = [0] * (max(a_neg) + 1)
print(f"  a = {a_neg}, f 크기 = {len(f)}")
for i in range(len(a_neg)):
    f[a_neg[i]] += 1
    print(f"  a[{i}]={a_neg[i]} → f[{a_neg[i]}] += 1 → f = {f}")
print("  ⚠️ 에러가 안 났다! 파이썬 음수 인덱스가 '뒤에서부터'를 뜻하기 때문")
print("     → -3은 f[-3] 즉 뒤에서 3번째 칸을 건드린다. 조용히 틀린다 🔥")

print("\n[실수(float)라면?]")
try:
    f2 = [0] * 10
    f2[3.5] += 1
except TypeError as e:
    print(f"  TypeError: {e}  ← 이건 시끄럽게 실패")

print("\n[교재 실습 6-17이 입력을 제한하는 이유]")
print("  25~28행: while True: x[i] = int(input(...)); if x[i] >= 0: break")
print("  → 0 이상의 정수만 받도록 강제하고 있다.")

---
# 2️⃣ PART 3 — 2단계: 누적 도수 분포표 (5~7번)

> 교재 298p [그림 6-39]. 여기가 도수 정렬에서 **가장 안 와닿는 부분**이야. 천천히 가자.

### 5. 🟢 [손] 누적 도수 분포표를 직접 채워보자

```python
for i in range(1, max + 1):
    f[i] += f[i - 1]
```

1단계 결과: `f = [1, 1, 1, 2, 1, 1, 0, 1, 0, 0, 1]`

**한 줄씩 직접 채워봐.**

| 실행 | 하는 일 | f (0~10) |
|---|---|---|
| 시작 | | `[1, 1, 1, 2, 1, 1, 0, 1, 0, 0, 1]` |
| i=1 | `f[1] += f[0]` | `[1, 2, 1, 2, 1, 1, 0, 1, 0, 0, 1]` |
| i=2 | `f[2] += f[1]` | `[_,_,_,_,_,_,_,_,_,_,_]` |
| i=3 | | |
| i=4 | | |
| i=5 | | |
| i=6 | | |
| i=7 | | |
| i=8 | | |
| i=9 | | |
| i=10 | | |

**완성된 누적 도수 분포표 f = `[_, _, _, _, _, _, _, _, _, _, _]`**

- 왜 `i`가 **0이 아니라 1부터** 시작할까? `i=0`이면 `f[0] += f[-1]`이 되는데 무슨 일이 벌어져?
- 마지막 값 `f[10]`은 뭐가 나왔어? 그 값의 의미는?

*(답을 적은 뒤 실행)*

In [ ]:
f = [1, 1, 1, 2, 1, 1, 0, 1, 0, 0, 1]
mx = 10
print("1단계 결과")
show_arr(f, "f")
for i in range(1, mx + 1):
    f[i] += f[i - 1]
    print(f"\nf[{i}] += f[{i-1}]")
    show_arr(f, "f", mark={i})

print(f"\n완성된 누적 도수 분포표 f = {f}")
print(f"f[{mx}] = {f[mx]} = n  ← 10점 이하인 학생 = 전체 학생")

### 6. 🟡 [설명] 누적 도수가 대체 무슨 뜻이야

교재 298p: **"'0점부터 n점까지 학생이 몇 명 있는지'를 누적된 값을 나타내는 누적 도수 분포표 f를 만듭니다."**
그리고 각주: **"f[4]의 값인 6은 0~4점을 받은 학생의 누계가 6명이고, f[10]의 값 9는 0~10점을 받은 학생의 누계가 9명이라는 것을 의미합니다."**

완성된 표: `f = [1, 2, 3, 5, 6, 7, 7, 8, 8, 8, 9]`

**표를 채워봐.**

| f[i] | 값 | 의미 (말로) |
|---|---|---|
| f[0] | 1 | 0점 이하인 학생이 1명 |
| f[3] | ① | ② |
| f[5] | ③ | ④ |
| f[6] | ⑤ | ⑥ 6점 학생은 0명인데 왜 f[6]=f[5]야? |
| f[10] | ⑦ | ⑧ |

**그리고 여기가 핵심** 🔥 — 이 숫자를 **"몇 명"이 아니라 "몇 번째 자리"** 로 다시 읽어봐.

- `f[3] = 5` 는 "3점 이하가 5명"이야. 그럼 정렬된 배열에서 **3점인 학생 중 마지막 사람**은 몇 번째 자리에 있어야 해? (1부터 세면 5번째)
- 배열 인덱스는 0부터니까, 그 자리의 **인덱스**는 ⑨____ 야.
- 즉 **`f[v]` 는 "값 v가 들어갈 마지막 자리의 다음 인덱스"** 라고 볼 수 있어. 교재 299p가 `f[a[i]] -= 1` 을 먼저 하는 이유가 바로 이거야.

*(답을 적은 뒤 실행)*

In [ ]:
f = [1, 2, 3, 5, 6, 7, 7, 8, 8, 8, 9]
srt = sorted(SAMPLE)
print("정렬된 결과 (미리 보기)")
show_arr(srt, "b")
print()
print(" v  | f[v] | 의미                  | 값 v가 차지하는 인덱스 구간")
print("----+------+-----------------------+----------------------------")
for v in range(len(f)):
    lo = f[v-1] if v > 0 else 0
    hi = f[v]
    span = f"b[{lo}]~b[{hi-1}]" if hi > lo else "(없음)"
    print(f" {v:2d} |  {f[v]:2d}  | {v}점 이하가 {f[v]}명{'':8s}| {span}")

### 7. 🔴 [설명] 누적을 안 하면 안 될까

**사고 실험**: 2단계를 통째로 건너뛰고, 1단계 도수 분포표만으로 정렬하려면?

`f = [1, 1, 1, 2, 1, 1, 0, 1, 0, 0, 1]` (누적 전)

방법이 하나 있긴 해:
```python
idx = 0
for v in range(max + 1):
    for _ in range(f[v]):     # v를 f[v]번 반복해서 쓴다
        a[idx] = v; idx += 1
```

- 이 코드도 정렬은 돼. 실행해서 확인해봐.
- 그런데 **교재는 왜 이 방법을 안 쓸까?** 🔥
  💡 힌트: 학생 데이터가 `(점수, 이름)` 이라면? 위 방법은 `a[idx] = v` 로 **점수만** 써넣지. **이름은 어디로 갔어?**
- 시간 복잡도도 따져봐. 이중 `for`처럼 보이지만 안쪽 총 반복 횟수는? 겉보기와 실제가 다르지?
- 결론: 누적 도수를 만드는 이유는 **"각 원소를 통째로, 정확한 위치에 옮기기 위해"** 야. 값만 재생성하는 게 아니라 **원본 원소를 이동**시키는 거지.

*(답을 적은 뒤 실행)*

In [ ]:
# 누적 없이 '값 재생성' 방식
a1 = SAMPLE[:]
mx = max(a1)
f1 = [0] * (mx + 1)
for v in a1: f1[v] += 1
out = []
for v in range(mx + 1):
    for _ in range(f1[v]): out.append(v)
print("값 재생성 방식:", out, "→ 정렬 성공?", out == sorted(SAMPLE))

# 그런데 (점수, 이름) 이라면?
print("\n[원소가 (점수, 이름)일 때]")
students = [(5,'현수'), (7,'현진'), (0,'범관'), (2,'영현'), (5,'은성')]
f2 = [0] * 8
for s in students: f2[s[0]] += 1
out2 = []
for v in range(8):
    for _ in range(f2[v]): out2.append(v)
print("  값 재생성 방식:", out2, "  ← 이름이 전부 사라졌다! ❌")

def fsort_pair(a, mx):
    n = len(a); f = [0]*(mx+1); b = [None]*n
    for i in range(n): f[a[i][0]] += 1
    for i in range(1, mx+1): f[i] += f[i-1]
    for i in range(n-1, -1, -1):
        f[a[i][0]] -= 1; b[f[a[i][0]]] = a[i]
    for i in range(n): a[i] = b[i]

t = students[:]
fsort_pair(t, 7)
print("  도수 정렬 방식:", t, "  ← 원소가 통째로 이동 ✅")

---
# 3️⃣ PART 4 — 3단계: 작업용 배열 만들기 (8~11번)

> 교재 299~300p [그림 6-40, 6-41, 6-42]. 오늘의 하이라이트야.

```python
for i in range(n - 1, -1, -1):
    f[a[i]] -= 1
    b[f[a[i]]] = a[i]
```

두 줄뿐인데 **감소를 먼저 하고 저장을 나중에** 한다는 게 핵심이야.

### 8. 🟢 [손] 그림 6-40 — 첫 번째 원소 배치

```
a = [5, 7, 0, 2, 4, 10, 3, 1, 3]     (인덱스 0~8)
f = [1, 2, 3, 5, 6, 7, 7, 8, 8, 8, 9]  (누적 도수, 인덱스 0~10)
b = [_, _, _, _, _, _, _, _, _]      (작업용, 전부 비어 있음)
```

**`i = 8` 부터 시작** (배열 a의 **맨 끝**부터!)

- `a[8]` = ①____
- 현재 `f[①]` = ②____   → 교재: **"0~3점 사이에 학생이 5명 있다는 의미입니다."**
- **먼저 감소**: `f[①] -= 1` → `f[①]`이 ③____ 가 됨
- **그 다음 저장**: `b[③] = ①` → **b[____]에 값 ____ 저장**

```
b = [_, _, _, _, ⑤, _, _, _, _]
f = [1, 2, 3, ⑥, 6, 7, 7, 8, 8, 8, 9]
```

- 교재 각주: **"배열 b의 다섯 번째 원소 인덱스는 5가 아닌 4라는 점을 주의합니다."**
  → "다섯 번째"와 "인덱스 4"의 차이를 설명해봐. 이게 `-= 1` 을 먼저 하는 이유와 어떻게 연결돼?

*(답을 적은 뒤 실행)*

In [ ]:
a = SAMPLE[:]
n = len(a)
f = [1, 2, 3, 5, 6, 7, 7, 8, 8, 8, 9]
b = [None] * n

print("시작 상태")
show_arr(a, "a"); show_arr(f, "f"); show_arr(b, "b")

i = 8
v = a[i]
print(f"\ni = {i}, a[{i}] = {v}")
print(f"  현재 f[{v}] = {f[v]}   ← 0~{v}점 사이에 학생이 {f[v]}명")
f[v] -= 1
print(f"  ① f[{v}] -= 1  →  f[{v}] = {f[v]}")
b[f[v]] = v
print(f"  ② b[f[{v}]] = a[{i}]  →  b[{f[v]}] = {v}")
print()
show_arr(f, "f", mark={v}); show_arr(b, "b", mark={f[v]})

### 9. 🟡 [손] 그림 6-41, 6-42 — 계속 진행

8번에 이어서 `i = 7`, `i = 6` 을 직접 해봐.

**i = 7**: `a[7]` = ①____
- `f[①]` = ②____ → 감소 후 ③____ → `b[③] = ①`
- 교재 300p: **"f[1]의 값 2는 0~1점 사이에 학생이 2명 있다는 것을 보여 줍니다. 그러므로 작업용 배열 b[1]에 1을 저장합니다."**

**i = 6**: `a[6]` = ④____  ← 🔥 **3점이 또 나왔다!**
- `f[3]`은 지금 얼마야? (8번에서 5 → 4로 줄여놨지)
- 감소 후 ⑤____ → `b[⑤] = 3`
- 교재 300p: **"3점인 학생은 [그림 6-40]에서도 이미 저장했으므로 두 번째 하는 것입니다. ... 이렇게 미리 값을 감소시켰기 때문에 중복되는 값인 3을 배열 b[3]에 저장할 수 있습니다."**

**표를 채워봐:**

| i | a[i] | 감소 전 f[a[i]] | 감소 후 | b의 어느 칸에 | b 상태 |
|---|---|---|---|---|---|
| 8 | 3 | 5 | 4 | b[4] | `[_,_,_,_,3,_,_,_,_]` |
| 7 | ① | ② | ③ | b[③] | |
| 6 | ④ | | ⑤ | b[⑤] | |
| 5 | | | | | |
| 4 | | | | | |
| 3 | | | | | |
| 2 | | | | | |
| 1 | | | | | |
| 0 | | | | | |

**최종 b = `[_, _, _, _, _, _, _, _, _]`**

*(끝까지 채운 뒤 실행)*

In [ ]:
a = SAMPLE[:]
n = len(a)
f = [1, 2, 3, 5, 6, 7, 7, 8, 8, 8, 9]
b = [None] * n

print(" i | a[i] | f 감소전→후 |  저장  | b")
print("---+------+-------------+--------+" + "-"*40)
for i in range(n - 1, -1, -1):
    v = a[i]
    before = f[v]
    f[v] -= 1
    b[f[v]] = v
    shown = "[" + ", ".join("_" if x is None else str(x) for x in b) + "]"
    print(f" {i} |  {v:2d}  |  f[{v}] {before}→{f[v]}   | b[{f[v]}]={v} | {shown}")

print(f"\n최종 b = {b}")
print("정렬 성공?", b == sorted(SAMPLE))

### 10. 🔴 [설명] 🔥 왜 **감소를 먼저** 하고 저장을 나중에 할까

```python
f[a[i]] -= 1        # ① 먼저 감소
b[f[a[i]]] = a[i]   # ② 그 다음 저장
```

순서를 바꿔서 이렇게 쓰면 안 될까?
```python
b[f[a[i]]] = a[i]   # 먼저 저장
f[a[i]] -= 1        # 나중에 감소
```

- 6번에서 봤듯 `f[3] = 5` 는 **"3점 이하가 5명"** 이야. 그럼 3점인 학생 중 마지막 사람의 **인덱스**는 몇이지? `5`야, `4`야?
- 1부터 세는 "몇 번째"와 0부터 세는 "인덱스" 사이의 **1 차이**를 메우는 게 바로 `-= 1` 이야.
- 순서를 바꾸면 어떤 일이 벌어질까? **예측하고** 실행해봐.
  - 에러가 날까? 조용히 틀릴까?
  - 3점이 두 명인데, 두 명이 **같은 칸**에 들어가진 않을까?
- 교재 300p: **"작업용 배열 b에 값을 저장할 때 참조한 배열 f의 원솟값을 1 감소시킨 이유는 같은 값의 원소를 중복으로 처리하지 않기 위한 것입니다."**
  → **"중복으로 처리하지 않는다"** 는 게 정확히 무슨 뜻인지 설명해봐.

*(예측을 적은 뒤 실행)*

In [ ]:
def fsort_wrong_order(a, mx):
    """저장을 먼저, 감소를 나중에 (잘못된 순서)"""
    n = len(a); f = [0]*(mx+1); b = [None]*n
    for i in range(n): f[a[i]] += 1
    for i in range(1, mx+1): f[i] += f[i-1]
    for i in range(n-1, -1, -1):
        b[f[a[i]]] = a[i]      # 🐛 먼저 저장
        f[a[i]] -= 1           # 🐛 나중에 감소
    return b

try:
    res = fsort_wrong_order(SAMPLE[:], max(SAMPLE))
    print("결과:", res)
except IndexError as e:
    print(f"IndexError: {e}")
    print("→ f[a[i]]가 n과 같은 값일 수 있어서 b의 범위를 벗어난다!")

print("\n[구체적으로 무슨 일이?]")
print(f"  f[10] = 9 인데 b의 크기는 {len(SAMPLE)} → b[9]는 존재하지 않는다")
print(f"  즉 f[v]는 '개수'이지 '인덱스'가 아니다. 인덱스로 쓰려면 반드시 -1 이 필요.")

### 11. 🟡 [설명] 왜 **맨 끝에서 앞으로** 스캔할까

```python
for i in range(n - 1, -1, -1):    # n-1 → 0 (역순)
```

- 앞에서부터(`range(n)`) 스캔해도 **정렬 결과는 똑같이 맞아.** 실행해서 확인해봐.
- 그런데 교재 303p에 이렇게 적혀 있어:
  > **"각 단계(for 문)에서 배열 원소를 건너뛰지 않고 순서대로 스캔하므로 이 정렬 알고리즘은 안정적입니다. 그러나 3단계에서 배열 a를 스캔할 때 맨 앞부터 스캔하면 안정적이지 않다는 점을 주의해야 합니다."**
- **왜 방향이 안정성을 결정할까?** 🔥
  - `f[v]` 는 값 v가 들어갈 자리 중 **가장 뒤쪽 인덱스 + 1** 이야.
  - `-= 1` 을 반복하면 자리가 **뒤에서 앞으로** 채워져.
  - 그럼 원본 배열도 **뒤에서 앞으로** 읽어야 순서가 보존되겠지?
- `a = [5, 7, 0, 2, 4, 10, 3, 1, 3]` 에서 3이 두 개야 (`a[6]`과 `a[8]`).
  - 역순 스캔: `a[8]` → b[4], `a[6]` → b[3]. 즉 **원래 앞에 있던 `a[6]`이 b에서도 앞(b[3])** ✅
  - 정순 스캔: `a[6]` → b[4], `a[8]` → b[3]. **순서가 뒤집힘** ❌
  - 교재 303p 각주가 정확히 이 얘기야.

*(답을 적은 뒤 실행)*

In [ ]:
def fsort_dir(a, mx, forward=False):
    """forward=True면 앞→뒤 스캔"""
    n = len(a); f = [0]*(mx+1); b = [None]*n
    for i in range(n): f[a[i][0]] += 1
    for i in range(1, mx+1): f[i] += f[i-1]
    rng = range(n) if forward else range(n-1, -1, -1)
    for i in rng:
        f[a[i][0]] -= 1; b[f[a[i][0]]] = a[i]
    return b

# 원소에 '원래 위치' 라벨을 붙여서 추적
labeled = [(v, f"a{i}") for i, v in enumerate(SAMPLE)]
print("입력:", [(v, lab) for v, lab in labeled])
print("       3이 두 개: a6, a8\n")

for fwd, name in ((False, "역순 (교재)   "), (True, "정순 (바꿈)   ")):
    res = fsort_dir(labeled[:], max(SAMPLE), fwd)
    print(f"{name}: {[lab for _, lab in res]}")
    print(f"{'':16s}  값 {[v for v, _ in res]}")
    threes = [lab for v, lab in res if v == 3]
    print(f"{'':16s}  3점 학생 순서: {threes}  {'✅ 원래 순서 유지' if threes == ['a6','a8'] else '❌ 뒤집힘'}\n")

---
# 4️⃣ PART 5 — 4단계: 배열 복사 (12번)

### 12. 🟢 [설명] 왜 복사가 필요할까

```python
for i in range(n):
    a[i] = b[i]
```

교재 300p: **"정렬은 완료되었지만 정렬한 결과가 저장되는 것은 작업용 배열 b이므로 배열 a는 정렬하기 전의 상태입니다."**

- 3단계가 끝난 시점에 `a`와 `b`는 각각 어떤 상태야?
- 이 줄을 `a = b` 로 바꾸면 어떻게 될까? **예측하고** 실행해봐. 🔥
  💡 힌트: 4일차 "call by object reference", 3일차 "이름과 객체"
- `a[:] = b` 는 어떨까?
- 💡 23일차 병합 정렬에서도 `buff`라는 작업용 배열이 있었지. **차이점**은? (병합 정렬은 `buff`가 `n`칸이었고, 도수 정렬은 `b`가 `n`칸 + `f`가 `max+1`칸)

*(예측을 적은 뒤 실행)*

In [ ]:
def fsort_a_eq_b(a, mx):
    """a = b 로 바꾼 버전"""
    n = len(a); f = [0]*(mx+1); b = [0]*n
    for i in range(n): f[a[i]] += 1
    for i in range(1, mx+1): f[i] += f[i-1]
    for i in range(n-1, -1, -1):
        f[a[i]] -= 1; b[f[a[i]]] = a[i]
    a = b                      # 🐛 이름만 바꿔 끼움
    print(f"    (함수 안에서 본 a: {a})")

def fsort_slice(a, mx):
    """a[:] = b 버전"""
    n = len(a); f = [0]*(mx+1); b = [0]*n
    for i in range(n): f[a[i]] += 1
    for i in range(1, mx+1): f[i] += f[i-1]
    for i in range(n-1, -1, -1):
        f[a[i]] -= 1; b[f[a[i]]] = a[i]
    a[:] = b                   # 슬라이스 대입

for name, fn in (("교재 (a[i] = b[i])", fsort), ("a = b", fsort_a_eq_b), ("a[:] = b", fsort_slice)):
    x = SAMPLE[:]
    fn(x, max(x))
    print(f"  {name:22s} → 호출한 쪽의 x = {x}  {'✅' if x == sorted(SAMPLE) else '❌ 원본 그대로!'}")

print("\n→ a = b 는 함수 안의 지역 이름 a가 b를 가리키게 할 뿐,")
print("   호출한 쪽의 리스트 객체는 손도 대지 않는다. (4일차 call by object reference)")

---
# 💻 PART 6 — 코드 구현 (13~17번)

> 개념은 끝났어. 이제 실습 6-17을 직접 짠다.

### 13. 🟢 [설명] 네 개의 for, 네 개의 역할

```python
def fsort(a, max):
    n = len(a)
    f = [0] * (max + 1)
    b = [0] * n

    for i in range(n):              # A
        f[a[i]] += 1
    for i in range(1, max + 1):     # B
        f[i] += f[i - 1]
    for i in range(n - 1, -1, -1):  # C
        f[a[i]] -= 1; b[f[a[i]]] = a[i]
    for i in range(n):              # D
        a[i] = b[i]
```

**표를 채워봐.**

| for | 단계 | 하는 일 | 반복 횟수 | 오늘 몇 번 문제 |
|---|---|---|---|---|
| A | 1단계 | ① | ② | 3~4번 |
| B | 2단계 | ③ | ④ | 5~7번 |
| C | 3단계 | ⑤ | ⑥ | 8~11번 |
| D | 4단계 | ⑦ | ⑧ | 12번 |

- 네 개를 다 더하면 총 반복 횟수는 `3n + max` 야. 그래서 시간 복잡도가 **O(n + max)**.
- 교재 302p: **"프로그램에서는 단일 for 문만 사용하고 재귀 호출이나 이중 if 문이 없어 매우 효율이 좋은 알고리즘입니다."**
  → 지금까지 배운 정렬 중 **재귀도 없고 중첩 반복도 없는 것**이 또 있었나? 비교해봐.
- **A와 B의 순서를 바꾸면?** C와 D의 순서를 바꾸면? 각각 어떻게 될지 생각해봐.

*(답을 적은 뒤 실행)*

In [ ]:
n = len(SAMPLE); mx = max(SAMPLE)
print(f"n = {n}, max = {mx}")
print(f"  A: {n}회  (배열 a 전체 스캔)")
print(f"  B: {mx}회  (1 ~ max)")
print(f"  C: {n}회  (배열 a 역순 스캔)")
print(f"  D: {n}회  (배열 b 전체 복사)")
print(f"  합계: 3n + max = {3*n} + {mx} = {3*n + mx}회")
print(f"\n→ O(n + max)")
print(f"\n비교: n={n}인 데이터를 퀵 정렬하면 약 n log n = {n * 3.17:.0f}회")

### 14. 🟡 [빈칸] `fsort` 직접 구현

3~12번에서 뜯어본 걸 조립해.

**기대 출력**
```
[0, 1, 2, 3, 3, 4, 5, 7, 10]
랜덤 500회 검증: 실패 0회
```

In [ ]:
def my_fsort(a: MutableSequence, mx: int) -> None:
    """도수 정렬 (배열 원솟값은 0 이상 mx 이하)"""
    n = len(a)
    f = [0] * ___                    # ① 도수 분포표 크기
    b = [0] * ___                    # ② 작업용 배열 크기

    # [1단계] 도수 분포표
    for i in range(n):
        ___                          # ③

    # [2단계] 누적 도수 분포표
    for i in range(___, ___):        # ④⑤ 어디부터 어디까지?
        ___                          # ⑥

    # [3단계] 작업용 배열 만들기
    for i in range(___, ___, ___):   # ⑦⑧⑨ 방향 주의! (11번)
        ___                          # ⑩ 먼저 감소 (10번)
        ___                          # ⑪ 그 다음 저장

    # [4단계] 배열 복사
    for i in range(n):
        ___                          # ⑫ (12번 함정 주의)


def my_counting_sort(a: MutableSequence) -> None:
    """도수 정렬"""
    if len(a) > 0:
        my_fsort(a, ___)             # ⑬ max를 어떻게 구하지?


x = SAMPLE[:]
my_counting_sort(x)
print(x)

fail = 0
for _ in range(500):
    n = random.randint(0, 60)
    t = [random.randint(0, 40) for _ in range(n)]
    ref = sorted(t)
    my_counting_sort(t)
    if t != ref: fail += 1
print(f"랜덤 500회 검증: 실패 {fail}회")

### 15. 🟡 [디버깅] 빈 배열이 들어오면

```python
def counting_sort(a):
    fsort(a, max(a))
```

교재 실습 6-17의 `counting_sort`는 이렇게 되어 있어.

- `a = []` 를 넘기면 어떻게 될까? **에러 이름과 메시지를 예측**해봐.
- 그럼 어떻게 고쳐야 해?
- 💡 이런 종류의 엣지케이스를 어디서 또 봤지? (22일차 5번 `qsort(x)` + `num=0`, 24일차 `heap_sort([])`)
- 오늘 부록의 `counting_sort`는 이미 방어가 되어 있어. 어떻게 했는지 찾아봐.

*(예측을 적은 뒤 실행)*

In [ ]:
def counting_sort_raw(a):
    """교재 그대로 (방어 없음)"""
    fsort(a, max(a))

for name, arr in (("빈 배열 []", []), ("원소 1개 [7]", [7]), ("전부 0 [0,0,0]", [0,0,0])):
    try:
        t = arr[:]
        counting_sort_raw(t)
        print(f"  {name:18s} → {t}  ✅")
    except ValueError as e:
        print(f"  {name:18s} → ValueError: {e}  ❌")

print("\n부록 버전 (방어 있음)")
for name, arr in (("빈 배열 []", []), ("원소 1개 [7]", [7])):
    t = arr[:]
    counting_sort(t)
    print(f"  {name:18s} → {t}  ✅")

### 16. 🔴 [구현] 음수도 정렬하기

4번에서 봤듯 도수 정렬은 **음수를 처리하지 못해.** `f[-3]`이 조용히 엉뚱한 칸을 건드리거든.

**어떻게 고칠까?** 아이디어는 간단해 — 모든 값을 **최솟값만큼 밀어서** 0 이상으로 만드는 거야.

```
a = [3, -2, 5, -2, 0]
min = -2  →  전부 +2 하면  [5, 0, 7, 0, 2]   ← 이제 도수 정렬 가능!
정렬 후 다시 -2 하면 원래 값 복원
```

- 이때 `f` 배열의 크기는 몇이어야 해? (`max - min + 1`)
- 값을 실제로 바꾸지 않고 **인덱스 계산할 때만** 오프셋을 적용하는 게 더 깔끔해. 그렇게 짜봐.

**기대 출력**
```
[-5, -2, -2, 0, 1, 3, 7]
랜덤 500회 검증: 실패 0회
```

In [ ]:
def fsort_signed(a: MutableSequence) -> None:
    """음수를 포함한 정수 배열의 도수 정렬"""
    n = len(a)
    if n == 0: return
    lo, hi = min(a), max(a)
    size = ___                       # ① f 배열의 크기

    f = [0] * size
    b = [0] * n

    for i in range(n):
        f[___] += 1                  # ② 오프셋 적용

    for i in range(1, size):
        f[i] += f[i - 1]

    for i in range(n - 1, -1, -1):
        f[___] -= 1                  # ③
        b[f[___]] = a[i]             # ④

    for i in range(n):
        a[i] = b[i]


x = [3, -2, 5, -2, 0, 1, 7, -5]
fsort_signed(x)
print(x)

fail = 0
for _ in range(500):
    n = random.randint(0, 60)
    t = [random.randint(-30, 30) for _ in range(n)]
    ref = sorted(t)
    fsort_signed(t)
    if t != ref: fail += 1
print(f"랜덤 500회 검증: 실패 {fail}회")

### 17. 🔴 [구현] 원소가 튜플일 때 (실전 도수 정렬)

7번에서 확인했듯, 도수 정렬의 진가는 **원소를 통째로 옮길 때** 나와.

학생 데이터 `(점수, 이름)` 을 **점수 기준으로** 정렬해봐. 안정성이 유지되어야 해 (같은 점수면 원래 순서).

**기대 출력**
```
[(0, '범관'), (2, '영현'), (5, '현수'), (5, '은성'), (7, '현진')]
안정적인가? True
```

In [ ]:
def fsort_by_key(a: MutableSequence, mx: int, key) -> None:
    """key(원소)의 값(0~mx)을 기준으로 도수 정렬 (안정)"""
    n = len(a)
    f = [0] * (mx + 1)
    b = [None] * n

    for i in range(n):
        f[___] += 1                  # ① key 적용

    for i in range(1, mx + 1):
        f[i] += f[i - 1]

    for i in range(___, ___, ___):   # ② 안정성을 위한 방향
        f[___] -= 1                  # ③
        b[___] = a[i]                # ④ 원소를 '통째로' 저장

    for i in range(n):
        a[i] = b[i]


students = [(5, '현수'), (7, '현진'), (0, '범관'), (2, '영현'), (5, '은성')]
t = students[:]
fsort_by_key(t, 10, lambda s: s[0])
print(t)
print("안정적인가?", t == sorted(students, key=lambda s: s[0]))

---
# 📐 PART 7 — 성질과 제약 (18~20번)

> 교재 302~303p. **"이렇게 빠른데 왜 항상 쓰지 않을까?"** 에 답하는 파트야.

### 18. 🟡 [실험] 얼마나 빠른가

교재 302p: **"도수 정렬 알고리즘은 데이터 비교·교환 작업이 필요 없어 매우 빠릅니다."**

**먼저 예측해봐** — n = 200,000, 값 범위 0~100:

| 정렬 | 예상 순위 |
|---|---|
| 도수 (오늘) | |
| 퀵 (21일) | |
| 병합 (23일) | |
| 힙 (24일) | |
| 내장 `sort()` | |

- 도수 정렬이 **내장 `sort()`보다도 빠를까?** 내장은 C로 짜여 있는데?
- 값 범위가 **0~100**이라는 게 왜 도수 정렬에 유리한 조건이야?

*(예측을 적은 뒤 실행)*

In [ ]:
def quick_sort(a):
    def qs(a, l, r):
        pl, pr = l, r; x = a[(l+r)//2]
        while pl <= pr:
            while a[pl] < x: pl += 1
            while a[pr] > x: pr -= 1
            if pl <= pr: a[pl], a[pr] = a[pr], a[pl]; pl += 1; pr -= 1
        if l < pr: qs(a, l, pr)
        if pl < r: qs(a, pl, r)
    if a: qs(a, 0, len(a)-1)

def merge_sort(a):
    def _ms(a, l, r):
        if l < r:
            c = (l+r)//2; _ms(a, l, c); _ms(a, c+1, r)
            p = j = 0; i = k = l
            while i <= c: buff[p] = a[i]; p += 1; i += 1
            while i <= r and j < p:
                if buff[j] <= a[i]: a[k] = buff[j]; j += 1
                else: a[k] = a[i]; i += 1
                k += 1
            while j < p: a[k] = buff[j]; k += 1; j += 1
    n = len(a); buff = [None]*n; _ms(a, 0, n-1)

def heap_sort(a):
    def dh(a, left, right):
        temp = a[left]; parent = left
        while parent < (right+1)//2:
            cl = parent*2+1; cr = cl+1
            child = cr if cr <= right and a[cr] > a[cl] else cl
            if temp >= a[child]: break
            a[parent] = a[child]; parent = child
        a[parent] = temp
    n = len(a)
    for i in range((n-1)//2, -1, -1): dh(a, i, n-1)
    for i in range(n-1, 0, -1):
        a[0], a[i] = a[i], a[0]; dh(a, 0, i-1)

import sys
sys.setrecursionlimit(200000)
random.seed(1)
N = 200000
data = [random.randint(0, 100) for _ in range(N)]
print(f"[n = {N:,}, 값 범위 0~100]")
for nm, fn in (("도수(오늘)", counting_sort), ("퀵(21일)", quick_sort),
               ("병합(23일)", merge_sort), ("힙(24일)", heap_sort),
               ("내장 sort", lambda x: x.sort())):
    a = data[:]
    t = time.perf_counter(); fn(a); el = time.perf_counter() - t
    print(f"  {nm:12s}: {el:.4f}s  {'OK' if a == sorted(data) else 'X'}")

### 19. 🟡 [실험] 🔥 그런데 값 범위가 커지면

교재 302p: **"하지만 도수 분포표가 필요하므로 (예를 들어 0, 1, ..., 100점인 시험 점수와 같이) 데이터의 최솟값과 최댓값을 미리 알고 있는 경우에만 적용할 수 있습니다."**

이번엔 **n을 1,000으로 고정**하고 **값 범위만** 키워볼 거야.

**예측해봐**:

| 값 범위 | 도수 정렬 시간 | 도수 정렬 메모리 | 퀵 정렬 시간 |
|---|---|---|---|
| 0 ~ 100 | | | |
| 0 ~ 10,000 | | | |
| 0 ~ 1,000,000 | | | |

- 데이터는 **1,000개뿐**인데 `f` 배열의 크기는 몇이 될까?
- 시간 복잡도 O(n + max) 에서 **max가 n보다 훨씬 커지면** 어떻게 되지?
- ⚠️ 값 범위가 0~1억이면 어떻게 될까? (실행하지 마 — 계산만 해봐. `f` 배열이 몇 MB가 될까?)

*(예측을 적은 뒤 실행 — 조금 걸려)*

In [ ]:
def quick_sort(a):
    def qs(a, l, r):
        pl, pr = l, r; x = a[(l+r)//2]
        while pl <= pr:
            while a[pl] < x: pl += 1
            while a[pr] > x: pr -= 1
            if pl <= pr: a[pl], a[pr] = a[pr], a[pl]; pl += 1; pr -= 1
        if l < pr: qs(a, l, pr)
        if pl < r: qs(a, pl, r)
    if a: qs(a, 0, len(a)-1)

random.seed(2)
print(f"[n = 1,000 고정, 값 범위만 변화]")
print(f"{'값 범위':>12s} | {'f 크기':>10s} | {'도수 시간':>10s} | {'도수 메모리':>12s} | {'퀵 시간':>9s}")
print("-" * 68)
for mx in (100, 10_000, 1_000_000):
    d = [random.randint(0, mx) for _ in range(1000)]
    a = d[:]
    tracemalloc.start()
    t = time.perf_counter(); counting_sort(a); el = time.perf_counter() - t
    cur, peak = tracemalloc.get_traced_memory(); tracemalloc.stop()
    a2 = d[:]
    t = time.perf_counter(); quick_sort(a2); el2 = time.perf_counter() - t
    print(f"{mx:12,d} | {mx+1:10,d} | {el:9.4f}s | {peak/1024:10.1f}KB | {el2:8.4f}s")

print("\n[만약 값 범위가 0 ~ 1억이라면? (계산만)]")
size = 100_000_000 + 1
print(f"  f 배열 크기: {size:,}칸")
print(f"  파이썬 int 리스트는 원소당 약 8바이트 포인터 → 약 {size*8/1024/1024:,.0f} MB")
print(f"  정렬할 데이터는 겨우 1,000개인데! 💀")

### 20. 🟢 [정리] 정렬 8종 최종 총정리

이번 주로 교재 6장이 끝났어. 표를 완성해봐.

| 정렬 | 평균 | 최악 | 추가 메모리 | 안정? | 비교 기반? | 배운 날 |
|---|---|---|---|---|---|---|
| 버블 | O(n²) | O(n²) | O(1) | ✅ | ✅ | 18일 |
| 선택 | O(n²) | O(n²) | O(1) | ❌ | ✅ | 19일 |
| 삽입 | O(n²) | O(n²) | O(1) | ✅ | ✅ | 19일 |
| 셸 | ~O(n^1.25) | O(n²) | O(1) | ❌ | ✅ | 20일 |
| 퀵 | O(n log n) | O(n²) | O(log n) | ❌ | ✅ | 21~22일 |
| 병합 | O(n log n) | O(n log n) | O(n) | ✅ | ✅ | 23일 |
| 힙 | O(n log n) | O(n log n) | O(1) | ❌ | ✅ | 24일 |
| **도수** | ① | ② | ③ | ④ | ⑤ | **오늘** |

**최종 질문 3개**

1. 도수 정렬만 유일하게 **O(n log n)의 벽을 뚫었어.** 그게 가능한 근본 이유가 뭐야? 그리고 그 대가로 무엇을 포기했지?
2. **다음 세 상황에서 어떤 정렬을 고를래?** 이유와 함께.
   - (가) 100만 명의 시험 점수(0~100점)를 정렬
   - (나) 메모리가 극도로 부족한 임베디드 기기에서 최악을 보장해야 함
   - (다) `(이름, 나이)` 데이터를 나이순으로, 같은 나이면 입력 순서 유지
3. 파이썬 `sorted()`가 도수 정렬을 안 쓰는 이유는? (힌트: `sorted()`는 문자열도, 튜플도, 사용자 객체도 정렬할 수 있어야 해)

*(답을 적은 뒤 실행)*

In [ ]:
print("[비교 기반 정렬의 하한이 O(n log n)인 이유 — 직관]")
print("  원소 n개의 가능한 순서는 n! 가지.")
print("  비교 한 번은 후보를 최대 절반으로 줄임 → 최소 log2(n!) 번 필요")
print("  log2(n!) ≈ n log2(n) - 1.44n  →  Ω(n log n)")
import math
for n in (10, 100, 1000):
    print(f"    n={n:5d}: log2({n}!) ≈ {math.log2(math.factorial(n)):>10,.0f}   "
          f"n·log2(n) = {n*math.log2(n):>10,.0f}")

print("\n[도수 정렬이 이 하한을 피하는 이유]")
print("  비교를 한 번도 하지 않으므로 위 논증이 적용되지 않는다.")
print("  대신 '값이 곧 인덱스'라는 강한 가정을 전제로 한다.")
print("  → 정수여야 하고, 0 이상이어야 하고, 범위를 미리 알아야 한다.")

print("\n[(다) 상황 — 나이순 + 안정성]")
people = [('현수', 24), ('현진', 22), ('범관', 24), ('영현', 21), ('은성', 22)]
t = people[:]
def fsort_pair2(a, mx, kf):
    n = len(a); f = [0]*(mx+1); b = [None]*n
    for i in range(n): f[kf(a[i])] += 1
    for i in range(1, mx+1): f[i] += f[i-1]
    for i in range(n-1, -1, -1):
        f[kf(a[i])] -= 1; b[f[kf(a[i])]] = a[i]
    for i in range(n): a[i] = b[i]
fsort_pair2(t, 30, lambda p: p[1])
print("  도수 정렬:", t)
print("  sorted() :", sorted(people, key=lambda p: p[1]))
print("  일치?", t == sorted(people, key=lambda p: p[1]))

---
---

# ✅ 정답 & 해설

> ⚠️ **표를 직접 채운 뒤에 내려와.** 특히 3·5·9번은 손으로 써야 남아.

---

## 🔁 Remind

### R-1
- 다섯 줄의 공통점: **전부 두 원소를 비교하는 조건문**(`<`, `>`, `<=`, `>=`)이야. 원소의 값 자체가 아니라 **"둘 중 뭐가 큰가"** 만 물어봐.
- 그래서 **비교 정렬**. 하한이 O(n log n)인 이유는 20번 실행 셀에서 다뤄.
- 💡 비교를 안 하려면 **원소의 값 자체를 "위치 정보"로 쓸 수 있어야** 해. 그러려면 값이 정수여야 하고, 범위를 알아야 하지.

### R-2
- ① **✅ 안정** (단, 조건부)
- ② 원소를 **교환하지 않고** 순서대로 스캔해서 제자리에 배치하기 때문
- ⚠️ 교재 303p 함정: **3단계에서 배열 a를 맨 앞부터 스캔하면 안정적이지 않아.** 11번에서 실증해.

---

## 🎯 PART 1 해설

### 1. 비교 없이 정렬하기

- 방법 B에서 답안지끼리 **비교한 적이 없어.** 각 답안지는 자기 점수만 보고 자기 칸으로 가.
- 미리 알아야 하는 것: **값의 범위(최솟값~최댓값).** 칸을 몇 개 만들지 정해야 하니까.
- 키(cm)라면 약 100~250 → **칸 151개** (충분히 현실적)
  연봉이라면 0~수억 → **칸 수억 개** (비현실적) 💀
  실수(3.14)라면 → **칸을 만들 수 없음.** 3.14와 3.15 사이에 무한히 많은 값이 있으니까.
- 이 세 예가 도수 정렬의 적용 범위를 정확히 보여줘. **"이산적이고, 범위가 좁고, 미리 알 수 있는 값"** 일 때만 쓸 수 있어.

---

### 2. 도수 분포표

- **"등급"** = 점수 값 (0점, 1점, …, 10점) → 배열 `f`의 **인덱스**
- **"도수"** = 그 점수를 받은 학생 수 → 배열 `f`의 **값**
- `f[7] = 1` → 7점인 학생이 1명. `f[6] = 0` → **6점인 학생이 한 명도 없음**
- `f`의 크기는 **`max + 1`**. 인덱스 0부터 max까지 총 `max+1`개가 필요하니까. 크기를 `max`로 만들면 실행 결과처럼 `f[10]`에서 `IndexError`가 나.

> 🔑 **값이 인덱스가 되고, 개수가 값이 된다.** 데이터의 역할이 뒤바뀌는 게 도수 정렬의 출발점이야.

---

## 1️⃣ PART 2 해설

### 3. 도수 분포표 채우기

| i | a[i] | f |
|---|---|---|
| 0 | 5 | `[0,0,0,0,0,1,0,0,0,0,0]` |
| 1 | 7 | `[0,0,0,0,0,1,0,1,0,0,0]` |
| 2 | 0 | `[1,0,0,0,0,1,0,1,0,0,0]` |
| 3 | 2 | `[1,0,1,0,0,1,0,1,0,0,0]` |
| 4 | 4 | `[1,0,1,0,1,1,0,1,0,0,0]` |
| 5 | 10 | `[1,0,1,0,1,1,0,1,0,0,1]` |
| 6 | 3 | `[1,0,1,1,1,1,0,1,0,0,1]` |
| 7 | 1 | `[1,1,1,1,1,1,0,1,0,0,1]` |
| 8 | 3 | `[1,1,1,2,1,1,0,1,0,0,1]` |

**완성: `f = [1, 1, 1, 2, 1, 1, 0, 1, 0, 0, 1]`**

- `f[3] = 2` — a에 3이 `a[6]`, `a[8]` 두 번 나왔으니까.
- `f[6] = f[8] = f[9] = 0` — 그 점수를 받은 학생이 없음. **값이 없어도 칸은 존재해야 해.**
- **f의 모든 원소를 더하면 n(=9)**. 모든 학생이 정확히 한 칸에만 세어지니까. 이건 **검산 도구**로도 써먹을 수 있어.

---

### 4. `f[a[i]] += 1`

- 대괄호 두 겹: `a[i]` 먼저 평가 → `f[5] += 1`
- 값이 인덱스로 변신하는 걸 **"값을 주소로 쓴다"** 또는 **직접 주소 지정(direct addressing)** 이라고 해.
- 비교가 필요 없어진 이유: 값 5는 **자기가 갈 칸을 스스로 알고 있어.** 다른 원소에게 물어볼 필요가 없지.
- 조건: ① **정수여야 함** ② **0 이상이고 max 이하여야 함**

**음수 실험 🔥**
```
a = [5, -3, 2], f 크기 = 6
a[1]=-3 → f[-3] += 1 → f = [0,0,0,1,0,1]
⚠️ 에러가 안 났다!
```
파이썬 음수 인덱스는 "뒤에서부터"를 뜻해서 `f[-3]`이 `f[3]`을 건드려. **조용히 틀리는 버그** — 22일차 10번, 23일차 3번과 같은 계열이야.

실수는 `TypeError: list indices must be integers` 로 **시끄럽게** 실패해. 이쪽이 오히려 안전하지.

교재 실습 6-17의 25~28행이 `if x[i] >= 0: break` 로 입력을 제한하는 이유가 바로 이거야.

---

## 2️⃣ PART 3 해설

### 5. 누적 도수 분포표 채우기

| 실행 | f |
|---|---|
| 시작 | `[1, 1, 1, 2, 1, 1, 0, 1, 0, 0, 1]` |
| i=1 | `[1, 2, 1, 2, 1, 1, 0, 1, 0, 0, 1]` |
| i=2 | `[1, 2, 3, 2, 1, 1, 0, 1, 0, 0, 1]` |
| i=3 | `[1, 2, 3, 5, 1, 1, 0, 1, 0, 0, 1]` |
| i=4 | `[1, 2, 3, 5, 6, 1, 0, 1, 0, 0, 1]` |
| i=5 | `[1, 2, 3, 5, 6, 7, 0, 1, 0, 0, 1]` |
| i=6 | `[1, 2, 3, 5, 6, 7, 7, 1, 0, 0, 1]` |
| i=7 | `[1, 2, 3, 5, 6, 7, 7, 8, 0, 0, 1]` |
| i=8 | `[1, 2, 3, 5, 6, 7, 7, 8, 8, 0, 1]` |
| i=9 | `[1, 2, 3, 5, 6, 7, 7, 8, 8, 8, 1]` |
| i=10 | `[1, 2, 3, 5, 6, 7, 7, 8, 8, 8, 9]` |

**완성: `f = [1, 2, 3, 5, 6, 7, 7, 8, 8, 8, 9]`**

- **i가 1부터 시작하는 이유**: `f[0]`은 이미 "0점 이하의 누계"라서 더할 게 없어. 그리고 `i=0`이면 `f[0] += f[-1]` 이 되는데, 파이썬에서 `f[-1]`은 **마지막 원소**(=1)라서 조용히 엉뚱한 값을 더해버려. 4번의 음수 인덱스 함정이 여기서도 나와!
- `f[10] = 9 = n`. **10점 이하인 학생 = 전체 학생**이니 당연해. 이것도 검산 도구야.

---

### 6. 누적 도수의 의미 🔥

| f[i] | 값 | 의미 |
|---|---|---|
| f[0] | 1 | 0점 이하가 1명 |
| f[3] | ① **5** | ② 3점 이하가 5명 |
| f[5] | ③ **7** | ④ 5점 이하가 7명 |
| f[6] | ⑤ **7** | ⑥ 6점 이하도 7명 — **6점 학생이 0명**이라 누계가 그대로 |
| f[10] | ⑦ **9** | ⑧ 10점 이하가 9명 = 전체 |

**핵심 재해석**
- `f[3] = 5` → 정렬된 배열에서 3점인 학생 중 마지막 사람은 **5번째**(1부터 셀 때)
- 인덱스는 0부터니까 ⑨ **4**
- 즉 **`f[v]` 는 "값 v가 들어갈 마지막 자리의 인덱스 + 1"**

실행 결과의 구간 표를 보면 명확해:
```
 v=3: f[3]=5, 구간 b[3]~b[4]   ← 3이 두 칸 차지
 v=6: f[6]=7, 구간 (없음)      ← 6점 학생 없음
```

> 🔑 **누적 도수는 "몇 명"이자 동시에 "몇 번째 자리까지"다.** 이 이중 해석이 도수 정렬의 마법이야.

---

### 7. 누적을 안 하면

- 값 재생성 방식도 **정렬 자체는 성공**해. `[0,1,2,3,3,4,5,7,10]`
- 그런데 원소가 `(점수, 이름)` 이면 **이름이 전부 사라져.** `a[idx] = v` 는 점수만 쓰니까.
```
값 재생성 방식: [0, 5, 5, 7]      ← 이름 소실 ❌
도수 정렬 방식: [(0,'범관'), (5,'현수'), (5,'은성'), (7,'현진')]  ✅
```
- 시간 복잡도: 이중 `for`처럼 보이지만 안쪽 총 반복이 `sum(f) = n`이라 **O(n + max)** 로 같아. 겉모습에 속으면 안 돼.
- **결론**: 누적 도수를 쓰는 이유는 **원본 원소를 통째로 정확한 위치로 옮기기 위해서**야. 값만 재생성하는 게 아니라. 실무에서 도수 정렬을 쓰는 거의 모든 경우가 이쪽이야(레코드 정렬, 기수 정렬의 부품 등).

---

## 3️⃣ PART 4 해설

### 8. 그림 6-40

- ① = **3** (`a[8]`)
- ② = **5** (`f[3]`) — 0~3점이 5명
- ③ = **4** (감소 후)
- `b[4] = 3`, ⑤ = **3**, ⑥ = **4**

**"다섯 번째" vs "인덱스 4"**: 사람은 1부터 세고 배열은 0부터 세. 그 **1 차이**를 메우는 게 `-= 1` 이야. 교재 각주가 굳이 이걸 짚은 이유지.

---

### 9. 그림 6-41, 6-42

| i | a[i] | f 감소 전→후 | 저장 | b |
|---|---|---|---|---|
| 8 | 3 | 5→4 | b[4]=3 | `[_,_,_,_,3,_,_,_,_]` |
| 7 | 1 | 2→1 | b[1]=1 | `[_,1,_,_,3,_,_,_,_]` |
| 6 | 3 | 4→3 | b[3]=3 | `[_,1,_,3,3,_,_,_,_]` 🔥 |
| 5 | 10 | 9→8 | b[8]=10 | `[_,1,_,3,3,_,_,_,10]` |
| 4 | 4 | 6→5 | b[5]=4 | `[_,1,_,3,3,4,_,_,10]` |
| 3 | 2 | 3→2 | b[2]=2 | `[_,1,2,3,3,4,_,_,10]` |
| 2 | 0 | 1→0 | b[0]=0 | `[0,1,2,3,3,4,_,_,10]` |
| 1 | 7 | 8→7 | b[7]=7 | `[0,1,2,3,3,4,_,7,10]` |
| 0 | 5 | 7→6 | b[6]=5 | `[0,1,2,3,3,4,5,7,10]` |

**최종 b = `[0, 1, 2, 3, 3, 4, 5, 7, 10]`** ✅

**i=6이 핵심** 🔥 — 3점이 두 번째로 나왔는데, `f[3]`이 이미 5→4로 줄어 있어서 이번엔 **b[3]** 에 들어가. 두 3이 서로 다른 칸을 차지하는 거야.

---

### 10. 🔥 왜 감소를 먼저 하나

- `f[3] = 5` 는 "3점 이하가 5명" = **개수**야. 마지막 3점 학생의 **인덱스**는 `5 - 1 = 4`.
- `f[v]` 는 개수이지 인덱스가 아니야. 인덱스로 쓰려면 **반드시 -1** 이 필요해.

**순서를 바꾸면**
```
IndexError: list assignment index out of range
```
`f[10] = 9` 인데 `b`의 크기는 9라서 `b[9]`가 존재하지 않아. 다행히 **시끄럽게 실패**해.

**"중복으로 처리하지 않는다"의 의미**: 같은 값이 여러 개일 때, 첫 번째를 저장한 뒤 `f[v]`를 줄여놓지 않으면 두 번째도 **똑같은 칸**에 덮어써져. 그럼 원소 하나가 사라지고 빈 칸이 생기지. 감소가 곧 **"이 자리는 이제 찼다"는 표시**야.

> 🔑 `f[v]` 는 카운터에서 **"다음에 쓸 자리를 가리키는 포인터"** 로 역할이 바뀌어. 한 배열이 3단계에 걸쳐 세 가지 의미(도수 → 누적 → 커서)를 갖는 게 이 알고리즘의 압축미야.

---

### 11. 🔥 왜 뒤에서부터 스캔하나

- 정순으로 해도 **정렬 결과는 맞아.** 하지만 **안정성이 깨져.**

**실측**
```
역순 (교재): ['a2','a7','a3','a6','a8','a4','a0','a1','a5']
             3점 학생 순서: ['a6', 'a8']  ✅ 원래 순서 유지
정순 (바꿈): ['a2','a7','a3','a8','a6','a4','a0','a1','a5']
             3점 학생 순서: ['a8', 'a6']  ❌ 뒤집힘
```

**논리**: `f[v]`가 가리키는 건 **가장 뒤쪽 자리**이고, `-= 1`로 앞으로 채워 나가. 채우는 방향이 **뒤→앞**이니, 읽는 방향도 **뒤→앞**이어야 원래 순서가 보존돼.

교재 303p 각주가 정확히 이 얘기야: 정순으로 하면 원래 앞에 있던 `a[6]`의 3이 `b[4]`(뒤)로, 뒤에 있던 `a[8]`의 3이 `b[3]`(앞)으로 가서 **같은 키값의 순서 관계가 뒤바뀌어.**

> 🔑 23일차 병합 정렬은 `<=` **한 글자**가 안정성을 결정했고, 오늘은 `for`문의 **방향**이 결정해. 안정성은 늘 아주 작은 디테일에 달려 있어.

---

## 4️⃣ PART 5 해설

### 12. 배열 복사

- 3단계 후: `b`는 정렬 완료, `a`는 **원본 그대로**야.
- `a = b` 로 바꾸면? **실측**
```
교재 (a[i] = b[i])     → x = [0,1,2,3,3,4,5,7,10]  ✅
a = b                  → x = [5,7,0,2,4,10,3,1,3]  ❌ 원본 그대로!
a[:] = b               → x = [0,1,2,3,3,4,5,7,10]  ✅
```
- `a = b`는 **함수 안의 지역 이름 `a`가 `b`를 가리키게 할 뿐**, 호출한 쪽의 리스트 객체는 손도 안 대. 4일차 **call by object reference** 그대로야. 함수 안에서는 정렬된 것처럼 보여서 더 헷갈리지.
- `a[:] = b`는 **슬라이스 대입**이라 원본 객체의 내용을 바꿔. 이건 동작해.
- **23일차 병합 정렬과 비교**: 병합은 `buff` **n칸**만 썼는데, 도수 정렬은 `b` **n칸** + `f` **max+1칸**. 즉 추가 메모리가 **O(n + max)** 야. max가 크면 이게 치명타가 돼 (19번).

---

## 💻 PART 6 해설

### 13. 네 개의 for

| for | 단계 | 하는 일 | 반복 |
|---|---|---|---|
| A | 1 | ① 각 값의 개수를 센다 | ② n |
| B | 2 | ③ 앞 칸을 더해 누적으로 만든다 | ④ max |
| C | 3 | ⑤ a를 역순 스캔하며 b의 제자리에 배치 | ⑥ n |
| D | 4 | ⑦ b를 a로 복사 | ⑧ n |

총 `3n + max` → **O(n + max)**

- **재귀도 없고 중첩 반복도 없는 정렬**: 지금까지 배운 것 중 도수 정렬이 유일해. 버블/선택/삽입/셸은 이중 반복, 퀵/병합은 재귀, 힙은 이중 반복(내부 while)이 있어.
- **A와 B를 바꾸면**: 아직 세지도 않은 빈 표를 누적하니 전부 0. 완전히 망가져.
- **C와 D를 바꾸면**: 아직 채우지도 않은 `b`(전부 0)를 `a`에 복사한 뒤 그 `a`로 배치하니 역시 망가져. **네 단계는 순서가 절대적**이야.

---

### 14. `my_fsort`

```python
f = [0] * (mx + 1)                    # ①
b = [0] * n                           # ②
for i in range(n):
    f[a[i]] += 1                      # ③
for i in range(1, mx + 1):            # ④⑤
    f[i] += f[i - 1]                  # ⑥
for i in range(n - 1, -1, -1):        # ⑦⑧⑨  ← 역순! (11번)
    f[a[i]] -= 1                      # ⑩  ← 감소 먼저 (10번)
    b[f[a[i]]] = a[i]                 # ⑪
for i in range(n):
    a[i] = b[i]                       # ⑫  ← a = b 아님! (12번)
my_fsort(a, max(a))                   # ⑬
```
출력: `[0, 1, 2, 3, 3, 4, 5, 7, 10]`, 랜덤 500회 실패 0회

---

### 15. 빈 배열

```
빈 배열 []       → ValueError: max() arg is an empty sequence  ❌
원소 1개 [7]     → [7]  ✅
전부 0 [0,0,0]   → [0, 0, 0]  ✅
```

- `max([])` 가 터져. 고치는 법: `if len(a) > 0:` 로 감싸기.
- 22일차 5번(`num=0`일 때 `a[-1]`), 24일차(빈 배열 힙 정렬)와 **같은 계열의 엣지케이스**야. 이번 주 내내 반복되는 패턴이지 — **"길이 0을 넣어봤나?"** 는 체크리스트에 넣어둘 것.

---

### 16. 음수 지원

```python
size = hi - lo + 1                    # ①
f[a[i] - lo] += 1                     # ②
f[a[i] - lo] -= 1                     # ③
b[f[a[i] - lo]] = a[i]                # ④  ← 저장하는 값은 원본 a[i]!
```
출력: `[-5, -2, -2, 0, 1, 3, 5, 7]`, 랜덤 500회 실패 0회

**포인트**: 인덱스 계산할 때만 `- lo` 를 적용하고, **`b`에 저장하는 값은 원본 `a[i]`** 야. 값을 실제로 바꿔놨다가 나중에 되돌리는 것보다 깔끔하고 실수가 적어.

💡 보너스: `size = hi - lo + 1` 이라 **값의 범위가 좁으면** 값이 아무리 커도 괜찮아. 예를 들어 `[1000000, 1000002, 1000001]` 은 size가 3이야!

---

### 17. 튜플 정렬

```python
f[key(a[i])] += 1                     # ①
for i in range(n - 1, -1, -1):        # ②  ← 안정성
    f[key(a[i])] -= 1                 # ③
    b[f[key(a[i])]] = a[i]            # ④  ← 원소 통째로
```
출력: `[(0,'범관'), (2,'영현'), (5,'현수'), (5,'은성'), (7,'현진')]`, 안정 True

**7번에서 예고한 그것**이야. `b[...] = a[i]` 로 **원소를 통째로** 옮기는 게 값 재생성 방식과의 결정적 차이. 현수(5점)와 은성(5점)의 입력 순서가 그대로 유지되지.

> 💡 이 구조가 **기수 정렬(radix sort)** 의 부품이야. 자릿수마다 도수 정렬을 반복하는데, 매번 **안정적이어야만** 전체가 성립해. 그래서 역순 스캔이 필수인 거야.

---

## 📐 PART 7 해설

### 18. 얼마나 빠른가

**실측 (n = 200,000, 값 범위 0~100)**
```
도수(오늘)   : 0.043s   ← 파이썬 구현인데 1위권
퀵(21일)     : 0.201s
병합(23일)   : 0.370s
힙(24일)     : 0.485s
내장 sort    : 0.024s
```

- 도수 정렬은 **퀵보다 4.6배, 힙보다 11배** 빨라.
- 내장 `sort()`(C 구현)에는 못 미치지만 **같은 자릿수**야. 23일차 4번에서 "파이썬 루프는 C보다 수십 배 느리다"고 했는데, 여기선 2배 차이밖에 안 나. **알고리즘 우위가 언어 차이를 거의 상쇄**한 거지.
- **0~100이 유리한 이유**: max=100이 n=200,000보다 훨씬 작아서 `O(n + max) ≈ O(n)` 이 돼. `f` 배열도 101칸뿐이라 메모리도 공짜에 가깝고.
- 그리고 값이 20만 개인데 종류는 101가지뿐 → **중복이 엄청 많아.** 비교 정렬은 중복이 많아도 여전히 n log n번 비교하지만, 도수 정렬은 중복이 많을수록 유리해.

---

### 19. 🔥 값 범위가 커지면

**실측 (n = 1,000 고정)**
```
     값 범위 |     f 크기 |  도수 시간 |  도수 메모리 |   퀵 시간
         100 |        101 |   0.0019s |     11.2KB |  0.0007s
      10,000 |     10,001 |   0.0186s |    320.8KB |  0.0008s
   1,000,000 |  1,000,001 |   2.0689s |  31,005.2KB |  0.0006s   💀
```

- 데이터는 **1,000개뿐**인데 값 범위가 100만이면 `f`가 **100만 칸**, 메모리 **30MB**, 시간은 퀵보다 **3,400배 느려.**
- `O(n + max)` 에서 max가 n을 압도하면 사실상 **O(max)** 가 돼. n은 아무 의미가 없어지지.
- 값 범위 0~1억이면 `f` 배열이 약 **763 MB**. 정렬할 데이터는 1,000개인데! (실제로 이 셀을 돌리면 메모리 부족으로 프로세스가 죽어)

> 🔑 **도수 정렬의 성능은 n이 아니라 max가 지배한다.** "n이 크고 max가 작을 때"만 이기는 알고리즘이야. 반대 조건에선 최악의 선택이 돼.

**적용 판단 기준**: 대략 **max ≲ n** 이면 고려할 만하고, **max ≫ n** 이면 쓰면 안 돼.

---

### 20. 정렬 8종 최종 총정리

| 정렬 | 평균 | 최악 | 추가 메모리 | 안정? | 비교 기반? |
|---|---|---|---|---|---|
| 버블 | O(n²) | O(n²) | O(1) | ✅ | ✅ |
| 선택 | O(n²) | O(n²) | O(1) | ❌ | ✅ |
| 삽입 | O(n²) | O(n²) | O(1) | ✅ | ✅ |
| 셸 | ~O(n^1.25) | O(n²) | O(1) | ❌ | ✅ |
| 퀵 | O(n log n) | O(n²) | O(log n) | ❌ | ✅ |
| 병합 | O(n log n) | O(n log n) | O(n) | ✅ | ✅ |
| 힙 | O(n log n) | O(n log n) | O(1) | ❌ | ✅ |
| **도수** | ① **O(n + max)** | ② **O(n + max)** | ③ **O(n + max)** | ④ **✅** (역순 스캔 시) | ⑤ **❌** |

**최종 질문 답**

**1.** 비교 기반 정렬의 하한 논증은 **"비교 한 번이 정보를 최대 1비트 준다"** 에 기반해. n!가지 순서를 구별하려면 최소 log₂(n!) ≈ n log n 번 필요하지. 도수 정렬은 **비교를 한 번도 안 하므로 이 논증이 적용되지 않아.**
포기한 것: **범용성.** 정수여야 하고, 범위를 미리 알아야 하고, 범위가 좁아야 해. 그리고 **메모리 O(max)** 를 지불하지.

**2.**
- **(가) 100만 명의 0~100점** → **도수 정렬.** n=100만 ≫ max=100. 완벽한 조건이야.
- **(나) 메모리 부족 + 최악 보장** → **힙 정렬.** 유일하게 "추가 메모리 O(1) + 최악 O(n log n)"을 동시에 만족해 (24일차 21번). 도수는 O(max) 메모리라 탈락, 병합은 O(n) 메모리라 탈락, 퀵은 최악 O(n²)라 탈락.
- **(다) (이름, 나이) 나이순 + 안정** → **도수 정렬** (나이는 0~120 정도로 범위가 좁으니 최적) 또는 **병합 정렬** (범위 제약 없이 안정). 실행 결과에서 도수 정렬로 정확히 처리되는 걸 확인했지.

**3.** `sorted()`는 **문자열, 튜플, 사용자 정의 객체** 등 무엇이든 정렬해야 해. 이들은 **"값을 인덱스로 변환"할 방법이 없어.** `f['현수'] += 1` 은 불가능하잖아. 비교(`__lt__`)만이 모든 타입에 통하는 유일한 인터페이스라, 범용 정렬은 반드시 비교 기반이어야 해.

---

## 📌 핵심 3줄 요약

1. **도수 정렬은 "값을 인덱스로" 쓴다.** 비교를 한 번도 안 하기 때문에 비교 정렬의 하한 O(n log n)을 뚫고 **O(n + max)** 를 달성해. 대가는 "정수 + 0 이상 + 좁은 범위를 미리 알 것"이라는 강한 전제야.
2. **배열 `f` 하나가 세 가지 역할을 한다.** 1단계에선 **도수**(각 값의 개수), 2단계에선 **누적 도수**(이 값 이하가 몇 명), 3단계에선 **커서**(다음에 쓸 자리). `f[a[i]] -= 1` 을 먼저 하는 건 "개수"를 "인덱스"로 바꾸는 -1 보정이자, 중복 값을 서로 다른 칸에 넣는 장치야.
3. **안정성은 `for`문의 방향에 달려 있다.** `f[v]`가 뒤쪽 자리부터 앞으로 채워지므로, 원본도 **뒤에서 앞으로** 읽어야 순서가 보존돼. 정순으로 바꾸면 정렬은 맞지만 안정성이 깨져 — 23일차의 `<=` 한 글자와 같은 계열의 디테일이야.

## 🗂️ 스터디 진행 가이드

- 🟢 **(R-1, 1, 2, 3, 5, 8, 12, 13, 15, 20번)**: 전원 필수
  - **3번 + 5번**을 손으로 채우는 게 오늘의 기본기. 이거 안 하면 8~11번이 전부 안 보여
  - **12번 `a = b` 함정**은 4일차 call by object reference의 실전 복습
- 🟡 **(R-2, 4, 6, 9, 11, 14, 18, 19번)**: 팀 목표선
  - **6번(누적 도수의 이중 해석)** 이 오늘의 개념 고비 🔥 — "몇 명"이자 "몇 번째 자리"
  - **9번**은 표 전체를 끝까지 채울 것. 3점 두 개가 서로 다른 칸에 들어가는 순간이 하이라이트
  - **19번 실측**은 꼭 돌려볼 것. 도수 정렬이 3,400배 느려지는 걸 봐야 제약이 체감돼
- 🔴 **(7, 10, 16, 17번)**: 도전
  - **10번이 오늘 최고 난도** 🔥🔥 — "왜 감소가 먼저인가"를 개수 vs 인덱스로 설명할 수 있으면 완전 이해
  - **17번**은 기수 정렬(radix sort)로 가는 다리야
- **금요일 코딩테스트 범위**: 🟢🟡 (R-1 ~ 20번)

## 🔗 오늘 회수된 개념들

- **19~24일차 비교 정렬 전체** → "비교하지 않는다"의 대조군 (R-1, 20번)
- **4일차 음수 인덱스 / 22일차 `(0-1)//2`** → `f[-3]`이 조용히 틀리는 함정 (4번), `f[0] += f[-1]` (5번)
- **4일차 call by object reference** → `a = b` vs `a[:] = b` vs `a[i] = b[i]` (12번)
- **23일차 병합 정렬 `buff`** → 작업용 배열이라는 같은 아이디어, 메모리 비용 비교 (12번)
- **23일차 `<=` 한 글자** → 오늘은 `for`문 방향이 안정성을 결정 (11번)
- **22일차 5번 / 24일차 빈 배열** → `max([])` 엣지케이스, 이번 주 반복 패턴 (15번)
- **24일차 힙 정렬 메모리 O(1)** → (나) 상황에서 힙이 정답인 이유 (20번)

---

> 🎉 **06장 정렬 알고리즘 완주!** 18일차 버블부터 오늘 도수까지 **8종**을 전부 손으로 짰어.
>
> **다음 진도**: 07장 **집합**, 또는 커리큘럼상 **해시 테이블** — 오늘 배운 "값을 인덱스로 쓴다"가 해시의 출발점이야.
> 도수 정렬은 값 자체를 인덱스로 썼지만, 해시는 **값을 함수에 통과시켜** 인덱스를 만들어. 그래서 값 범위가 아무리 커도 작은 배열에 담을 수 있어 — 오늘 19번에서 본 그 문제를 정면으로 해결하는 거야.